In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
import time
import port_for
import subprocess

# Env
import gymnasium as gym
from gymnasium import spaces
import torch

# Algo
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="6"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
seed = 316
random.seed(seed)
np.random.seed(seed)

In [4]:
def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    return "%d:%02d:%02d" % (hour, minutes, seconds)

# Load Data & Model

In [5]:
# Load data
# train_sequences and test_sequences are dictionaries like user_sequences
# Keys: UserIDs (ints)
# Values: list of tuples, each tuple is (movieID, rating)

# Dictionary maps for printing
with open('../data/idx2user.pkl', 'rb') as f:
    idx2user = pickle.load(f)
with open('../data/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)

with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)
with open('../data/test_sequences_idx.pkl', 'rb') as f:
    test_sequences_idx = pickle.load(f)

# ratings_dict is a Dictionary
# Each key is a tuple (user index, movie index)
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)

user_embeddings = np.load('../data/user_embeddings.npy')
movie_embeddings = np.load('../data/movie_embeddings.npy')

# Dictionary that maps Movie Index to Title for Quick Lookup
with open('../data/movie_index_to_title.pkl', 'rb') as f:
    movie_index_to_title = pickle.load(f)

# Dictionary that maps MovieID to Title for Quick Lookup
with open('../data/movie_id_to_title.pkl', 'rb') as f:
    movie_id_to_title = pickle.load(f)

# Env

In [6]:
class EvalPPORecommendationEnv(gym.Env):
    def __init__(self, 
                 train_sequences_idx,      # dict: user_idx -> [movie_idx, ...]
                 test_sequences_idx,       # dict: user_idx -> [movie_idx, ...]
                 ratings_dict,             # (user_idx, movie_idx) -> rating
                 user_embeddings, movie_embeddings,
                 evaluable_users,
                 n_history=5, T_max=10):
        super().__init__()
        self.train = train_sequences_idx
        self.test = test_sequences_idx
        self.ratings = ratings_dict
        self.user_embeddings = user_embeddings
        self.movie_embeddings = movie_embeddings
        self.evaluable_users = evaluable_users
        self.n_history = n_history
        self.T_max = T_max

        self.k = movie_embeddings.shape[1]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(3*self.k,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.k,), dtype=np.float32)
        self.item_weights = np.ones(self.n_history)

        self.current_user = None
        self.history = None
        self.candidate_items = None
        self.recommended_set = None
        self.step_count = 0

    def _get_state(self, user_idx, history_indices):
        u = self.user_embeddings[user_idx]
        weights = np.exp(self.item_weights) / np.sum(np.exp(self.item_weights))
        hist_embs = self.movie_embeddings[history_indices]
        weighted_avg = np.sum(hist_embs * weights[:, np.newaxis], axis=0)
        u_g = u * weighted_avg
        return np.concatenate([u, u_g, weighted_avg]).astype(np.float32)

    # def _get_state(self, user_idx, history_indices):
    #     u = self.user_embeddings[user_idx]
        
    #     # L = len(history_indices)
    #     # weights = np.exp(self.item_weights[:L])
    #     # weights = weights / np.sum(weights)
    #     L = len(history_indices)
    #     weights = np.ones(L) / L  # simple average
    
    #     hist_embs = self.movie_embeddings[history_indices]
    #     weighted_avg = np.sum(hist_embs * weights[:, np.newaxis], axis=0)
        
    #     u_g = u * weighted_avg
    #     return np.concatenate([u, u_g, weighted_avg]).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)

        # During evaluation, I'll always call reset(options={'user_idx': user_idx})
        # options lets me pass a specific user Index
        if options is not None and 'user_idx' in options:
            user_idx = options['user_idx']
        else:
            # NOTE: This should never happen b/c reset() has options passed in with user_idx
            # And the eval loop is iterating thru all user indices.
            user_idx = np.random.choice(self.evaluable_users)
        self.current_user = user_idx

        # Build positive history from training (last n_history items with rating >=4)
        pos_items = []
        for m in self.train[user_idx]:
            if self.ratings.get((user_idx, m), 0) >= 4:
                pos_items.append(m)
        self.history = pos_items[-self.n_history:].copy()

        # Set initial state
        self.candidate_items = set(self.test[user_idx])
        self.recommended_set = set()
        self.step_count = 0
        state = self._get_state(user_idx, self.history)
        return state, {}

    def step(self, action):
        candidates = list(self.candidate_items - self.recommended_set)
        if not candidates:
            next_state = self._get_state(self.current_user, self.history)
            return next_state, 0.0, True, False, {}

        cand_emb = self.movie_embeddings[candidates]
        scores = cand_emb @ action
        best_idx = np.argmax(scores)
        selected_item = candidates[best_idx]

        rating = self.ratings.get((self.current_user, selected_item), 0)
        reward = (rating - 3) / 2.0

        new_history = self.history.copy()
        if reward > 0:
            new_history = new_history[1:] + [selected_item]

        self.recommended_set.add(selected_item)
        self.step_count += 1

        terminated = (len(self.candidate_items - self.recommended_set) == 0)
        truncated = (self.step_count >= self.T_max)

        next_state = self._get_state(self.current_user, new_history)
        self.history = new_history

        info = {'selected_item': selected_item, 'rating': rating, 'reward': reward, 'step': self.step_count}
        return next_state, reward, terminated, truncated, info

In [7]:
def get_ground_truth(env, user_idx):
    """
        Returns a set of movie INDICES that the user rated positively (4 or 5)
    """
    truth = set()
    for movie_idx in env.test[user_idx]:
        if env.ratings.get((user_idx, movie_idx), 0) >= 4:
            truth.add(movie_idx)
    return truth

In [8]:
def evaluate_ppo(model, env, evaluable_users, top_k_list=[5, 10], max_steps=10, rating_threshold=4):    
    # Store all session trajectories
    all_sessions = {}
    metrics = {k: {'precision': [], 'ndcg': []} for k in top_k_list}
    user_IDs_to_explain = [1, 3, 40, 67, 316]

    for user_index in evaluable_users:
        user_id = idx2user[user_index]
        
        obs, _ = env.reset(options={'user_idx': user_index})
        recommended_items = []
        step = 0
        done = False

        # Get the list of movie indexes rated by the user index
        test_items = env.test[user_index]

        initial_history = env.history.copy()
        current_history = initial_history

        # Ground truth: positive items in test set (using env's test data)
        ground_truth = get_ground_truth(env, user_index)
        
        trajectory = {
            'user_id': idx2user[user_index],
            'user_idx': user_index,
            'initial_history': initial_history,
            'steps': [],
            'ground_truth': ground_truth,
            'test_seq': test_items
        }
        
        if user_id in user_IDs_to_explain:
            print("="*80)
            print(f"USER ID: {user_id} (Index: {user_index})")
            print(f"  Ground truth size: {len(ground_truth)}")
            print(f"Test Set Favorites (Ground Truth):")
            for movie_index in test_items:
                rating = ratings_dict[(user_index, movie_index)]
                if rating >= rating_threshold:
                    title = movie_index_to_title.get(movie_index, 'Unknown')
                    print(f"  - {title} (Actual Rating: {rating})")      

            print("Model Recommendations:")
            print(f"{'Rank':<5} | {'Title':<40} | {'Score':<10} | {'Hit?':<6} | {'Actual Rating':<6}")
            print("-" * 80)   

        # Run the session and SAVE the trajectory
        while not done and step < max_steps:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            if 'selected_item' in info:
                # info keys: 'selected_item', 'rating', 'reward', 'step'
                selected_item = info['selected_item']
                rating = info['rating']

                target_emb = env.movie_embeddings[selected_item]
                score = np.dot(action, target_emb)
                recommended_items.append(selected_item)
                
                # SAVE
                step_data = {
                    'step': step + 1,
                    'recommended_movie_idx': selected_item,
                    'recommended_movie_title': movie_index_to_title.get(selected_item, 'Unknown'),
                    'rating': rating,
                    'score': score,
                    'reward': reward,
                    'history_at_time': current_history.copy()
                }
                trajectory['steps'].append(step_data)

                if reward > 0:
                    current_history.append(selected_item)
                
                movie_title = movie_index_to_title.get(selected_item, "Unknown")
                if user_id in user_IDs_to_explain:
                    is_hit = "YES" if selected_item in ground_truth else "NO"
                    print(f"{step+1:<5} | {movie_title[:40]:<40} | {score: <10.4f}  | {is_hit:<6} | {rating:<6}")
            step += 1

        # Save trajectory
        all_sessions[user_index] = trajectory

        # Compute precision@k and NDCG@K for k=5 and k=10 for THIS current user
        for k in top_k_list:
            # Calculate precision@K
            top_k = recommended_items[:k]
            hits = sum(1 for m in top_k if m in ground_truth)
            precision = hits / k
            # Calculate NDCG@K
            dcg = sum(1 / np.log2(i+2) for i, m in enumerate(top_k) if m in ground_truth)
            idcg = sum(1 / np.log2(i+2) for i in range(min(len(ground_truth), k)))
            ndcg = dcg / idcg if idcg > 0 else 0.0
            # Store results
            metrics[k]['precision'].append(precision)
            metrics[k]['ndcg'].append(ndcg)

        # if user_id in user_IDs_to_explain:
            # print(f"    Precision@{k}: {metrics[k]['precision'][-1]:.4f}, NDCG@{k}: {metrics[k]['ndcg'][-1]:.4f}\n")

    # Outside the for loop now...
    # Calculate AVG. the precision@K & NDCG@K values for k=5 and k=10 FOR ALL USERS
    results = {}
    for k in top_k_list:
        results[f'Precision@{k}'] = np.mean(metrics[k]['precision'])
        results[f'NDCG@{k}'] = np.mean(metrics[k]['ndcg'])
    return all_sessions, results

# Eval

In [9]:
def get_evaluable_users(min_required=5):
# Create list of User Indices that you can legitimately evaluate
    evaluable_users = []
    for user_idx in train_sequences_idx.keys():
        # Count positive items in training
        pos_train = 0
        for movie_idx in train_sequences_idx[user_idx]:
            rating = ratings_dict.get((user_idx, movie_idx), 0)
            if rating >= 4:
                pos_train += 1
    
        # if # of positively rated movies in the train set < 5, skip the rest of the for loop
        if pos_train < min_required:
            continue
        
        # Check test set has at least one item
        test_items = test_sequences_idx.get(user_idx, [])
        if len(test_items) == 0:
            continue
        
        # Check test set has at least one positive item
        has_positive_test = False
        for movie_idx in test_items:
            rating = ratings_dict.get((user_idx, movie_idx), 0)
            if rating >= 4:
                has_positive_test = True
                break
        
        if not has_positive_test:
            continue
        
        evaluable_users.append(user_idx)

    return evaluable_users

In [58]:
all_Ab_study_eval_results = {}

history_lengths = [1, 3, 5, 7, 10]
episode_lengths = [10, 50, 100, 500]
ab_study_id = 1

version_number = 3
total_timesteps = 500_000

for n in history_lengths:
    evaluable_users = get_evaluable_users(n)
    print(f"Number of evaluable users: {len(evaluable_users)}")

    for t in episode_lengths:
        print(f"Ablation Study {ab_study_id}: n={n}, T_max={t}")
        model = PPO.load(f"../models/PPO_AbStudy_{ab_study_id}_n{n}_t{t}_Ver_{version_number}_{total_timesteps}")
        
        env = EvalPPORecommendationEnv(
            train_sequences_idx=train_sequences_idx,
            test_sequences_idx=test_sequences_idx,
            ratings_dict=ratings_dict,
            user_embeddings=user_embeddings,
            movie_embeddings=movie_embeddings,
            evaluable_users=evaluable_users,
            n_history=n,
            T_max=t
        )
        
        # Evaluate
        sessions, results = evaluate_ppo(model, env, evaluable_users, top_k_list=[5, 10], max_steps=10)
        print("PPO Ablation Study Evaluation on Test Set")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
            
        all_Ab_study_eval_results[ab_study_id] = {
            'n_history': n,
            'T_max': t,
            **results
        }
        # # Save the trajectory
        # with open(f'../results/PPO_AbStudy{ab_study_id}_n{n}_t{t}_session_trajectories.pkl', 'wb') as f:
        #     pickle.dump(sessions, f)

        ab_study_id += 1

Number of evaluable users: 5975
Ablation Study 1: n=1, T_max=10
USER ID: 1 (Index: 0)
  Ground truth size: 9
Test Set Favorites (Ground Truth):
  - Beauty and the Beast (1991) (Actual Rating: 5.0)
  - Aladdin (1992) (Actual Rating: 4.0)
  - Toy Story (1995) (Actual Rating: 5.0)
  - Bug's Life, A (1998) (Actual Rating: 5.0)
  - Antz (1998) (Actual Rating: 4.0)
  - Hunchback of Notre Dame, The (1996) (Actual Rating: 4.0)
  - Hercules (1997) (Actual Rating: 4.0)
  - Mulan (1998) (Actual Rating: 4.0)
  - Pocahontas (1995) (Actual Rating: 5.0)
Model Recommendations:
Rank  | Title                                    | Score      | Hit?   | Actual Rating
--------------------------------------------------------------------------------
1     | Bug's Life, A (1998)                     | 2.8508      | YES    | 5.0   
2     | Toy Story (1995)                         | 2.4570      | YES    | 5.0   
3     | Beauty and the Beast (1991)              | 2.4401      | YES    | 5.0   
4     | Aladdin (1992

In [59]:
print("Done!")

Done!


In [60]:
all_Ab_study_eval_results

{1: {'n_history': 1,
  'T_max': 10,
  'Precision@5': 0.6882343096234309,
  'NDCG@5': 0.7333096726453657,
  'Precision@10': 0.6072635983263599,
  'NDCG@10': 0.7615999084534475},
 2: {'n_history': 1,
  'T_max': 50,
  'Precision@5': 0.6752803347280335,
  'NDCG@5': 0.7147023634014746,
  'Precision@10': 0.5998493723849373,
  'NDCG@10': 0.7469747475391463},
 3: {'n_history': 1,
  'T_max': 100,
  'Precision@5': 0.6652384937238494,
  'NDCG@5': 0.7062748593792979,
  'Precision@10': 0.5937405857740586,
  'NDCG@10': 0.7408439369901353},
 4: {'n_history': 1,
  'T_max': 500,
  'Precision@5': 0.6732384937238494,
  'NDCG@5': 0.7156244855595488,
  'Precision@10': 0.5977740585774058,
  'NDCG@10': 0.7478531964491977},
 5: {'n_history': 3,
  'T_max': 10,
  'Precision@5': 0.6850209205020921,
  'NDCG@5': 0.7287425544471707,
  'Precision@10': 0.6018577405857741,
  'NDCG@10': 0.7561858680972547},
 6: {'n_history': 3,
  'T_max': 50,
  'Precision@5': 0.6796652719665272,
  'NDCG@5': 0.7219547269172222,
  'Preci

In [64]:
with open('../results/PPO_ablation_study_results.pkl', 'wb') as f:
    pickle.dump(all_Ab_study_eval_results, f)

# Print Ablation Study IDs of the best and worst metrics and each Study's metrics

In [61]:
best = {}
worst = {}

metrics = ["Precision@5", "Precision@10", "NDCG@5", "NDCG@10"]

for metric in metrics:
    best[metric] = max(all_Ab_study_eval_results.items(), key=lambda x: x[1][metric])
    worst[metric] = min(all_Ab_study_eval_results.items(), key=lambda x: x[1][metric])

In [63]:
for metric, (study_id, data) in best.items():
    # print(f"Best {metric}: {all_Ab_study_eval_results[metric]:.4f} (Study {study_id}, n={data['n_history']}, T={data['T_max']})")
    print(f"Best {metric}: {data[metric]:.4f} (Study {study_id}, n={data['n_history']}, T={data['T_max']})")

print("\n")

seen_ids = []
for _, (study_id, data) in best.items():
    if study_id not in seen_ids:
        print(f"Ablation Study {study_id}")
        for key in list(data.keys()):
            value = data[key]
            if isinstance(value, np.float64):
                value = round(value, 4)
            print(f"{key}: {value}")
        print("\n")
        seen_ids.append(study_id)

Best Precision@5: 0.6898 (Study 17, n=10, T=10)
Best Precision@10: 0.6129 (Study 17, n=10, T=10)
Best NDCG@5: 0.7333 (Study 1, n=1, T=10)
Best NDCG@10: 0.7616 (Study 1, n=1, T=10)


Ablation Study 17
n_history: 10
T_max: 10
Precision@5: 0.6898
NDCG@5: 0.7238
Precision@10: 0.6129
NDCG@10: 0.7524


Ablation Study 1
n_history: 1
T_max: 10
Precision@5: 0.6882
NDCG@5: 0.7333
Precision@10: 0.6073
NDCG@10: 0.7616




In [65]:
for metric, (study_id, data) in worst.items():
    print(f"Worst {metric}: {data[metric]:.4f} (Study {study_id}, n={data['n_history']}, T={data['T_max']})")

print("\n")

seen_ids = []
for _, (study_id, data) in worst.items():
    if study_id not in seen_ids:
        print(f"Ablation Study {study_id}")
        for key in list(data.keys()):
            value = data[key]
            if isinstance(value, np.float64):
                value = round(value, 4)
            print(f"{key}: {value}")
        print("\n")
        seen_ids.append(study_id)

Worst Precision@5: 0.6645 (Study 7, n=3, T=100)
Worst Precision@10: 0.5929 (Study 7, n=3, T=100)
Worst NDCG@5: 0.7029 (Study 7, n=3, T=100)
Worst NDCG@10: 0.7381 (Study 7, n=3, T=100)


Ablation Study 7
n_history: 3
T_max: 100
Precision@5: 0.6645
NDCG@5: 0.7029
Precision@10: 0.5929
NDCG@10: 0.7381


